# Phase 1 — Text → Video with Wan 2.1 on a free Kaggle GPU

**Goal:** prompt → 5-second MP4, using **ComfyUI + Wan 2.1 T2V 1.3B** on Kaggle's free T4 GPU.

ComfyUI runs in the background without its web page. This notebook talks to it through its **HTTP API**, which is exactly how the FastAPI backend will talk to it in Phase 3.

### Before you run
1. Top menu → **Settings → Accelerator → GPU T4 x2**
2. Top menu → **Settings → Internet → On** (needs a phone-verified Kaggle account)
3. Run the cells **top to bottom** (or *Run All*)

First run: about 5–10 minutes of setup (install + ~10 GB download), then the generation itself.

## Step 0 — Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print("PyTorch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("❌ No GPU. Top menu → Settings → Accelerator → GPU T4 x2, then run again.")
elif "P100" in torch.cuda.get_device_name(0):
    print("⚠️ P100 selected. Switch to GPU T4 x2 — newer PyTorch versions may not support the P100.")
else:
    print("✅", torch.cuda.get_device_name(0))

## Step 1 — Install ComfyUI

In [ ]:
import os, re, shutil, subprocess, sys

# Big files (ComfyUI + ~10 GB of models) go to scratch disk; finished videos go to /kaggle/working
def pick_workdir(min_free_gb=15):
    for d in ["/kaggle/temp", "/tmp", "/kaggle/working"]:
        if os.path.isdir(d) and shutil.disk_usage(d).free / 1e9 >= min_free_gb:
            return d
    raise RuntimeError("Not enough free disk space for the models (need ~15 GB)")

WORK = pick_workdir()
COMFY = f"{WORK}/ComfyUI"
OUT = "/kaggle/working/outputs"
LOG = f"{WORK}/comfyui.log"
os.makedirs(OUT, exist_ok=True)
print("Working folder:", WORK, f"({shutil.disk_usage(WORK).free / 1e9:.0f} GB free)")

if not os.path.isdir(COMFY):
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI "{COMFY}"

# Install ComfyUI's requirements, but keep Kaggle's own PyTorch (it already matches the GPU drivers)
reqs = open(f"{COMFY}/requirements.txt").read().splitlines()
reqs = [r for r in reqs if not re.match(r"^(torch|torchvision|torchaudio)\s*([<>=~!].*)?$", r.strip())]
open(f"{WORK}/comfy_requirements.txt", "w").write("\n".join(reqs))
!pip install -q -r "{WORK}/comfy_requirements.txt"
print("✅ ComfyUI installed")

## Step 2 — Download Wan 2.1 (≈10 GB, a few minutes)

| File | What it does |
|---|---|
| `wan2.1_t2v_1.3B_fp16` | The video diffusion model (1.3 billion parameters) |
| `umt5_xxl_fp8_e4m3fn_scaled` | Text encoder: turns your prompt into numbers the model understands |
| `wan_2.1_vae` | VAE: turns the model's compressed "latent" video into real frames |

In [ ]:
HF = "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files"
MODELS = [
    # (folder inside ComfyUI/models, file name, size)
    ("diffusion_models", "wan2.1_t2v_1.3B_fp16.safetensors", "~2.8 GB"),
    ("text_encoders", "umt5_xxl_fp8_e4m3fn_scaled.safetensors", "~6.7 GB"),
    ("vae", "wan_2.1_vae.safetensors", "~0.25 GB"),
]
for folder, name, size in MODELS:
    dest = f"{COMFY}/models/{folder}/{name}"
    if os.path.exists(dest):
        print("already downloaded:", name)
        continue
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    print(f"downloading {name} ({size}) ...")
    subprocess.run(["wget", "-q", "-c", "-O", dest + ".part", f"{HF}/{folder}/{name}"], check=True)
    os.rename(dest + ".part", dest)
print("✅ Models ready")

## Step 3 — Start ComfyUI in the background

In [ ]:
import json, time, urllib.request, urllib.error

API = "http://127.0.0.1:8188"

def comfy_ready():
    try:
        urllib.request.urlopen(f"{API}/system_stats", timeout=5)
        return True
    except Exception:
        return False

if not comfy_ready():
    server = subprocess.Popen(
        [sys.executable, "main.py", "--port", "8188"],
        cwd=COMFY, stdout=open(LOG, "w"), stderr=subprocess.STDOUT,
        env={**os.environ, "CUDA_VISIBLE_DEVICES": "0"},
    )
    for _ in range(150):  # wait up to ~5 minutes
        if comfy_ready() or server.poll() is not None:
            break
        time.sleep(2)

if comfy_ready():
    print("✅ ComfyUI is running (no web page needed — we use its API)")
else:
    print("❌ ComfyUI did not start. Last lines of its log:\n")
    print("".join(open(LOG).readlines()[-30:]))

## Step 4 — Your video settings

Start with the **quick 2-second test** (`FRAMES = 33`). When that works, set `FRAMES = 81` for the full 5 seconds and run Steps 4 and 5 again.

In [ ]:
PROMPT = ("A beautiful Indian bride walking through a palace corridor, cinematic lighting, "
          "slow camera movement, realistic photography")
NEGATIVE = ("blurry, low quality, distorted face, deformed hands, extra fingers, static image, "
            "watermark, text, subtitles, overexposed")
SEED = 42                  # same seed + same settings = same video; change it for a different take
WIDTH, HEIGHT = 832, 480   # 480p landscape. For a 9:16 reel: WIDTH, HEIGHT = 480, 832
FRAMES = 33                # Wan makes 16 frames per second: 33 ≈ 2 s (quick test), 81 = 5 s. Must be 4n+1
STEPS = 30                 # more steps = cleaner but slower

## Step 5 — Generate

Sends the workflow to ComfyUI, waits for the frames, then joins them into an MP4 with FFmpeg.
On a free T4 this takes **minutes, not seconds**: the 5-second version can take 20+ minutes.

In [ ]:
import csv
from IPython.display import Video, display

def build_workflow(prefix):
    """The ComfyUI workflow in API format: each node has a type, its settings,
    and links to other nodes written as [node_id, output_index]."""
    return {
        "1": {"class_type": "UNETLoader",  # loads the Wan video model
              "inputs": {"unet_name": "wan2.1_t2v_1.3B_fp16.safetensors", "weight_dtype": "default"}},
        "2": {"class_type": "ModelSamplingSD3",  # noise-schedule shift that Wan expects
              "inputs": {"model": ["1", 0], "shift": 8.0}},
        "3": {"class_type": "CLIPLoader",  # loads the text encoder
              "inputs": {"clip_name": "umt5_xxl_fp8_e4m3fn_scaled.safetensors", "type": "wan"}},
        "4": {"class_type": "CLIPTextEncode", "inputs": {"clip": ["3", 0], "text": PROMPT}},
        "5": {"class_type": "CLIPTextEncode", "inputs": {"clip": ["3", 0], "text": NEGATIVE}},
        "6": {"class_type": "VAELoader", "inputs": {"vae_name": "wan_2.1_vae.safetensors"}},
        "7": {"class_type": "EmptyHunyuanLatentVideo",  # empty video "canvas" in latent space
              "inputs": {"width": WIDTH, "height": HEIGHT, "length": FRAMES, "batch_size": 1}},
        "8": {"class_type": "KSampler",  # the diffusion loop: removes noise step by step
              "inputs": {"model": ["2", 0], "positive": ["4", 0], "negative": ["5", 0],
                         "latent_image": ["7", 0], "seed": SEED, "steps": STEPS, "cfg": 6.0,
                         "sampler_name": "uni_pc", "scheduler": "simple", "denoise": 1.0}},
        "9": {"class_type": "VAEDecode",  # latents → real frames
              "inputs": {"samples": ["8", 0], "vae": ["6", 0]}},
        "10": {"class_type": "SaveImage",  # saves every frame as a PNG
               "inputs": {"images": ["9", 0], "filename_prefix": prefix}},
    }

def comfy_alive():
    """True while the ComfyUI process is running. A busy ComfyUI can be too slow
    to answer HTTP requests, which doesn't mean it stopped."""
    proc = globals().get("server")
    return proc.poll() is None if proc is not None else comfy_ready()

def get_json(path):
    return json.load(urllib.request.urlopen(f"{API}{path}", timeout=60))

def still_queued(pid):
    try:
        queue = get_json("/queue")
    except Exception:
        return False
    return any(item[1] == pid for item in queue["queue_running"] + queue["queue_pending"])

if not comfy_alive():
    raise RuntimeError("ComfyUI is not running: run Step 3 again. Last log lines:\n"
                       + "".join(open(LOG).readlines()[-30:]))

# 1. Send the job (like POST /generate in the plan), unless the last one is still running
if globals().get("prompt_id") and still_queued(prompt_id):
    print("The last job is still running, waiting for it:", prompt_id)
    job_start = globals().get("job_start", globals().get("start", time.time()))
else:
    prefix = f"wan_t2v_{int(time.time())}"
    request = urllib.request.Request(
        f"{API}/prompt",
        data=json.dumps({"prompt": build_workflow(prefix)}).encode(),
        headers={"Content-Type": "application/json"},
    )
    try:
        prompt_id = json.load(urllib.request.urlopen(request))["prompt_id"]
    except urllib.error.HTTPError as e:
        raise RuntimeError("ComfyUI rejected the workflow:\n" + e.read().decode()) from None
    job_start = time.time()

# 2. Poll until it's finished (like GET /jobs/{id} in the plan)
log_start = os.path.getsize(LOG)
while True:
    try:
        history = get_json(f"/history/{prompt_id}")
        if prompt_id in history:
            break
    except Exception:
        pass  # ComfyUI can be slow to answer while the GPU is busy; ask again next time
    if not comfy_alive():
        raise RuntimeError("ComfyUI stopped. Last log lines:\n" + "".join(open(LOG).readlines()[-30:]))
    with open(LOG) as f:
        f.seek(log_start)
        bars = [l.strip() for l in f.read().replace("\r", "\n").splitlines() if "%|" in l]
    status = bars[-1][:90] if bars else "loading models / encoding prompt..."
    print(f"\r{int(time.time() - job_start):>5}s  {status}", end="", flush=True)
    time.sleep(10)

result = history[prompt_id]
if result.get("status", {}).get("status_str") == "error":
    raise RuntimeError("Generation failed:\n" + json.dumps(result["status"].get("messages"), indent=1)[-3000:])
frames = sorted(img["filename"] for img in result["outputs"]["10"]["images"])
minutes = (time.time() - job_start) / 60
print(f"\n✅ Generated {len(frames)} frames in {minutes:.1f} min")

# 3. Frames → MP4 with FFmpeg (16 fps is Wan's native frame rate)
ffmpeg = shutil.which("ffmpeg")
if ffmpeg is None:
    !pip install -q imageio-ffmpeg
    import imageio_ffmpeg
    ffmpeg = imageio_ffmpeg.get_ffmpeg_exe()
first = int(re.search(r"_(\d+)_\.png$", frames[0]).group(1))
mp4 = f"{OUT}/{prefix}.mp4"
subprocess.run([ffmpeg, "-y", "-loglevel", "error", "-framerate", "16", "-start_number", str(first),
                "-i", f"{COMFY}/output/{prefix}_%05d_.png",
                "-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", "18", mp4], check=True)

# 4. Record the run: generation time per setting is what you'll compare (and later monitor in MLOps)
runs_csv = f"{OUT}/runs.csv"
is_new = not os.path.exists(runs_csv)
with open(runs_csv, "a", newline="") as f:
    writer = csv.writer(f)
    if is_new:
        writer.writerow(["file", "prompt", "seed", "width", "height", "frames", "steps", "minutes"])
    writer.writerow([f"{prefix}.mp4", PROMPT, SEED, WIDTH, HEIGHT, FRAMES, STEPS, f"{minutes:.1f}"])

display(Video(mp4, embed=True, width=640))
print("Saved:", mp4, "→ download it from the Output section in the right panel")

## Next

- **Full 5 seconds:** set `FRAMES = 81` in Step 4, run Steps 4 and 5 again (the models stay loaded).
- **Experiment:** change `SEED`, the prompt, `STEPS`, or try 9:16 (`WIDTH, HEIGHT = 480, 832`). Every run is logged in `outputs/runs.csv`.
- **When you're done:** stop the session (power icon, top right) so you don't waste your weekly GPU hours.
- **If a cell fails:** copy the error message and ask for help.

### Phase 1 learning checklist
| Node in the workflow | Concept to learn |
|---|---|
| `CLIPLoader` + `CLIPTextEncode` | Text encoders, embeddings |
| `UNETLoader` | Diffusion transformer, model weights, fp16 vs fp8 |
| `EmptyHunyuanLatentVideo` | Latent space (video is generated compressed, not as pixels) |
| `ModelSamplingSD3` + `KSampler` | Noise, schedulers, steps, CFG, seeds |
| `VAEDecode` | VAE: latent → pixels |
| FFmpeg step | Frames → MP4, frame rate, codecs |